# 07 — Agent loop: planner + executor + budget + idempotent replay

**Goal:** trace one full agent turn end-to-end with a stubbed OpenAI client so the mechanics are visible without burning tokens. Then read the budget enforcement + plateau detector + circuit breaker.

Patterns mirror `017-sklearn-low-level/sklearn_agent/`: `(InputModel, fn, description)` tuple registry, `dispatch()` returns `{"error": ...}` instead of raising, idempotent replay keyed by `hash(name, args)`.

In [ ]:
from deepCab.agent import openai_tools, tool_names, dispatch
from deepCab.schemas.agent import BudgetCap, ImproveConfig

print(f'Tools ({len(tool_names())}):', tool_names())
schema = openai_tools()[0]
print('\nFirst tool schema preview:')
print(' name:', schema['function']['name'])
print(' description:', schema['function']['description'][:80], '...')
print(' parameter properties:', list(schema['function']['parameters'].get('properties', {})))


## Tool dispatch: errors are data, never exceptions

When the LLM picks a bad tool or sends malformed args, `dispatch` returns `{"error": "..."}`. The next LLM turn sees the error in the conversation and self-corrects — no exception unwinding the loop.

In [ ]:
print(dispatch('nope', {}))                     # unknown tool
print(dispatch('preprocess', {'data': 'bad'}))   # validation error


## A stubbed turn

We swap the OpenAI client for one that yields scripted tool-call messages. Verifies routing, the `{"error":...}` return, idempotent replay, and budget bookkeeping — no real model + no network.

In [ ]:
from dataclasses import dataclass, field
from typing import Any
import json

@dataclass
class _Fn: name: str; arguments: str
@dataclass
class _TC: id: str; function: Any
@dataclass
class _Msg: content: str | None = None; tool_calls: list = None  # type: ignore
@dataclass
class _Ch: message: _Msg
@dataclass
class _U:
    prompt_tokens: int = 100; completion_tokens: int = 50
    def model_dump(self): return {'prompt_tokens': 100, 'completion_tokens': 50}
@dataclass
class _R: choices: list; usage: Any = field(default_factory=_U)

class _Comp:
    def __init__(self, script): self.script = list(script)
    def create(self, **_kw):
        m = self.script.pop(0) if self.script else _Msg(content='done.')
        return _R(choices=[_Ch(message=m)])

class _Client:
    def __init__(self, script): self.chat = type('C', (), {'completions': _Comp(script)})()

client = _Client([
    _Msg(tool_calls=[_TC('c1', _Fn('list_runs', json.dumps({'top_k': 3})))]),
    _Msg(content='I see 0 prior runs — clean slate.')
])


In [ ]:
from deepCab.agent.budget import Budget
from deepCab.agent.executor import run_one_turn
from deepCab.agent.trace import AgentTrace

budget = Budget(cap=BudgetCap(max_iters=5, max_tool_calls=5, max_usd=1.0))
trace = AgentTrace()

events = list(run_one_turn(client, 'gpt-4o-mini', 'what runs do we have?',
                            budget, trace))
for ev in events:
    print(ev['event'], '->', {k: v for k, v in ev.items() if k != 'event'})

print(f'\nAfter turn: budget.tool_calls={budget.tool_calls}, usd=${budget.usd:.6f}')
print(f'Trace file: {trace.path}')


## Budget enforcement: atomic, deterministic

Each cap fires independently via a single `check_or_raise()`. `charge_llm_usage` books real token costs from the OpenAI usage response (not estimates). `restore(trace)` rebuilds totals from disk so resuming a loop doesn't double-count.

In [ ]:
from deepCab.agent.budget import Budget, BudgetExhausted

b = Budget(cap=BudgetCap(max_iters=2, max_tool_calls=3, max_usd=0.001))
b.charge_iter(); b.charge_iter()
try:
    b.check_or_raise()
except BudgetExhausted as e:
    print('blocked by:', e)


## Plateau detector — monotonic trend (P11 fix)

Pre-P11 used `max(window) - min(window) < eps` which lets oscillation pass. Replaced with mean-of-halves trend so [5,10,5,10] correctly counts as plateau.

In [ ]:
def plateau(window, eps):
    half = len(window)//2 or 1
    old = sum(window[:half]) / half
    new = sum(window[half:]) / max(len(window)-half, 1)
    improvement = old - new
    return improvement < eps, improvement

print('flat       ->', plateau([3.0]*4, 1e-3))
print('oscillate  ->', plateau([5, 10, 5, 10], 1e-3))
print('improving  ->', plateau([10, 9, 8, 7], 0.1))
print('worsening  ->', plateau([5, 6, 7, 8], 0.1))


## What's next

- `CONTRIBUTING.md → Adding a new agent tool` — 3 steps.
- `deepCab/agent/improve.py` — the outer loop: plan → execute → re-plan with circuit breaker.
